In [ ]:
from aiconfigurator.cli import cli_default

result = cli_default(
    model_path="Qwen/Qwen3-32B-FP8",
    backend="vllm",
    total_gpus=32,
    system="h200_sxm",
    tpot=15,
    save_dir="./output"
)

In [ ]:
import yaml
from pathlib import Path

output_dir = Path("manifests-patched/")
output_dir.mkdir(exist_ok=True)

matches = list(Path("output").glob("**/disagg/top1/k8s_deploy.yaml"))
assert matches, "No disagg/top1/k8s_deploy.yaml found under output/"
input_path = matches[0]

with input_path.open() as f:
    doc = yaml.safe_load(f)
    print(doc)

# manipulate
doc["metadata"]["namespace"] = "production"
doc["spec"]["replicas"] = 3

frontend = doc["spec"]["services"]["Frontend"]
# frontend.setdefault("extraPodSpec", {}).setdefault("mainContainer", {}).setdefault("env", [])
frontend["extraPodSpec"]["mainContainer"]["env"].append(
    {"name": "HF_HOME", "value": "/home/dynamo/.cache/huggingface"}
)

# save
out_path = output_dir / input_path.name
with out_path.open("w") as f:
    yaml.dump(doc, f, default_flow_style=False)